# 🔌 Model Context Protocol (MCP) Agent Protocols

### Overview & Learning Objectives
The **Model Context Protocol (MCP)** is an open standard designed to decouple LLM applications from data sources and tools. Rather than hardcoding custom APIs for each external capability, MCP establishes a client-server architecture where agents dynamically discover and consume:
- **Tools**: Executable functions (APIs, calculators, database querying).
- **Resources**: Structured contextual data (files, database tables, schemas).
- **Prompts**: Managed prompt templates stored directly on the server.

In this lab, we integrate LangChain with MCP using `langchain-mcp-adapters` and power our agent with Groq's **`llama-3.3-70b-versatile`** model across local and modular MCP servers.

## 📐 System Architecture: MCP Host, Client & Server

The diagram below visualizes the architectural hierarchy:

<div align="center">
  <img src="images/02_mcp_client_architecture.png" alt="Model Context Protocol (MCP) Client-Host-Server Architecture" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
flowchart TD
    subgraph Host [Agent Host Environment]
        Agent[🤖 LangChain ReAct Agent<br/>Groq llama-3.3-70b-versatile]
        Client[📦 MultiServerMCPClient]
        Agent <-->|Tool Binding & Execution| Client
    end

    subgraph Transport [Transport Layer]
        StdioLocal[Standard I/O Subprocess Pipe]
        StdioTime[UV / Subprocess Execution Pipe]
    end

    subgraph Servers [MCP Server Ecosystem]
        LocalSrv[💻 Local Server: 2.1_mcp_server.py<br/>Exposes: Tools, Resources, Prompts]
        TimeSrv[⏰ Time Server: mcp_server_time<br/>Exposes: Local Time Queries]
    end

    Client <-->|JSON-RPC Protocol| StdioLocal <--> LocalSrv
    Client <-->|JSON-RPC Protocol| StdioTime <--> TimeSrv
```

</details>


## 1. Environment Configuration

Load environment variables and ensure `GROQ_API_KEY` is present.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    print("⚠️ Warning: GROQ_API_KEY not detected. Please verify your .env file.")
else:
    print("✅ GROQ_API_KEY loaded successfully.")

## 2. Windows Async Subprocess Compatibility Patch

When running MCP servers over `stdio` transport on Windows inside Jupyter Notebooks (IPython kernel):
1. **ProactorEventLoop**: Must be explicitly set to enable asynchronous subprocess creation and communication pipes.
2. **Stderr Redirection**: Prevents `fileno()` stream descriptor errors caused by IPython's custom output wrapper.

In [ ]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__
    print("✅ Windows asyncio Proactor event loop configured.")

## 3. Local MCP Server Connection (`stdio`)

We initialize a `MultiServerMCPClient` pointing to our local server script (`resources/2.1_mcp_server.py`). The client spawns a background python process and connects via standard input/output streams using JSON-RPC.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "python",
            "args": ["resources/2.1_mcp_server.py"],
        }
    }
)
print("✅ MultiServerMCPClient initialized for local_server.")

## 4. Discovering Tools, Resources, and Prompts

MCP clients discover capabilities dynamically at runtime without prior hardcoding:
- `get_tools()`: Retrieves tool schemas conforming to LangChain BaseTool.
- `get_resources()`: Fetches context payloads.
- `get_prompt()`: Downloads standardized system instructions authored on the server.

In [ ]:
# Get tools exposed by the MCP server
tools = await client.get_tools()
print(f"Discovered Tools: {[t.name for t in tools]}")

# Get resources exposed by the MCP server
resources = await client.get_resources("local_server")
print(f"Discovered Resources: {len(resources)} resource(s)")

# Get parameterized prompt template
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content
print(f"Discovered System Prompt:\n{prompt}")

## 5. Compiling the Groq Agent with MCP Capabilities

We construct the agent using `ChatGroq(model="llama-3.3-70b-versatile")`, passing the dynamically discovered MCP tools and server-provided prompt.

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

groq_model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1
)

agent = create_agent(
    model=groq_model,
    tools=tools,
    system_prompt=prompt
)
print("✅ Groq Agent compiled with MCP tools.")

## 6. Execution Protocol: Sequence Diagram

<div align="center">
  <img src="images/seq_mcp_protocol.png" alt="MCP Protocol Execution Sequence Diagram" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
sequenceDiagram
    autonumber
    actor User
    participant Agent as Agent (ChatGroq)
    participant Client as MultiServerMCPClient
    participant Server as 2.1_mcp_server.py (Subprocess)

    User->>Agent: 'Tell me about the langchain-mcp-adapters library'
    Agent->>Agent: Decides to call discovered MCP tool
    Agent->>Client: ainvoke(tool_call)
    Client->>Server: JSON-RPC over stdio pipe: tools/call
    Server->>Server: Execute local tool function
    Server-->>Client: JSON-RPC response payload
    Client-->>Agent: ToolMessage(content=...)
    Agent->>Agent: Synthesize final answer
    Agent-->>User: Comprehensive explanation
```
</details>

## 7. Invoking the Agent Asynchronously

Because MCP communication uses async pipes, we execute the agent using `await agent.ainvoke(...)`.

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

print("Agent Final Answer:")
print("-" * 50)
print(response['messages'][-1].content)

## 8. Inspecting Response Payload

We inspect the full trace to verify the JSON-RPC tool call and return value.

In [ ]:
from pprint import pprint

pprint(response)

## 9. Modular MCP Servers: Time Server via `uv` / Subprocess

MCP servers can also be standalone Python packages executed on-the-fly with package managers like `uv` or installed modules. Below, we connect to `mcp_server_time` to give our Groq agent real-time clock awareness.

In [ ]:
time_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

time_tools = await time_client.get_tools()
print(f"Discovered Time Tools: {[t.name for t in time_tools]}")

## 10. Compiling & Testing the Time Agent

We compile a second Groq agent equipped with the newly discovered `time_tools`.

In [ ]:
time_agent = create_agent(
    model=groq_model,
    tools=time_tools,
)

question = HumanMessage(content="What time is it?")

time_response = await time_agent.ainvoke(
    {"messages": [question]}
)

print("Time Agent Response:")
print("-" * 50)
print(time_response['messages'][-1].content)